# Exploratory Data Analysis

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

# 01 — Data ingestion

## Purpose
Load the Zigong heart failure raw CSVs into Python, inspect the raw schema, fix errors, dtypes, etc

## Inputs
- `data/raw/zigong/dat.csv` — main clinical table
- `data/raw/zigong/dat_md.csv` — medication table
- `data/raw/zigong/dataDictionary.csv` — variable descriptions

## Dataset citation
Zhang Z et al. Sci Data. 2021;8(1):46.
https://doi.org/10.1038/s41597-021-00835-9

In [ ]:
# Define main data path
data_dir = Path("../data/raw/zigong")

# 1. Load CSV Files

In [ ]:
clinical = pd.read_csv(data_dir / "dat.csv", sep=",", index_col=0)
meds = pd.read_csv(data_dir / "dat_md.csv", sep=",", index_col=0)
dictionary = pd.read_csv(data_dir / "dataDictionary.csv", sep=",", index_col=0)

Helper Function 

In [ ]:
def dataframe_information(df: pd.DataFrame, name: str = "DataFrame") -> None:
    # Print name of Dataframe/Table
    print("=" * 50)
    print(f"  {name}")
    print("=" * 50)

    # Shape
    print(f"\nShape: {df.shape[0]} rows, {df.shape[1]} columns")

    # Column names
    print(f"\nColumns:\n{df.columns.tolist()}")

    # Data types
    print(f"\nData types:")
    print(df.dtypes)

    # Missing values
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(1)
    missing_df  = pd.DataFrame({
        "missing_n":   missing,
        "missing_pct": missing_pct
    }).query("missing_n > 0").sort_values("missing_pct", ascending=False)

    if missing_df.empty:
        print(f"\nMissing values: none")
    else:
        print(f"\nMissing values ({len(missing_df)} columns affected):")
        print(missing_df.to_string())

    # Duplicates
    dup_rows = df.duplicated().sum()
    print(f"\nDuplicate rows: {dup_rows}")

    # Numeric summary
    numeric_cols = df.select_dtypes(include=[np.number])
    if not numeric_cols.empty:
        print(f"\nNumeric summary ({len(numeric_cols.columns)} columns):")
        print(numeric_cols.describe().T.round(2).to_string())

    # Categorical summary
    cat_cols = df.select_dtypes(include=["object", "category"])
    if not cat_cols.empty:
        print(f"\nCategorical columns ({len(cat_cols.columns)} columns):")
        for col in cat_cols.columns:
            n_unique = df[col].nunique()
            top_vals = df[col].value_counts().head(3).to_dict()
            print(f"  {col}: {n_unique} unique — {top_vals}")

    print("\n")

## 2. Data dictionary

Always read the data dictionary before inspecting the data itself.
It tells us what columns mean, their expected units, and what outcome
variables are available.

In [ ]:
print(dictionary.columns.tolist())
dictionary

In [ ]:
# Find outcome / follow-up variables specifically
outcome_mask = dictionary.apply(
    lambda col: col.astype(str).str.contains(
        "death|mort|outcome|re_adm|readmit|follow|discharge",
        case=False,
        na=False,
    )
).any(axis=1)

print("Potential outcome / follow-up variables:")
dictionary[outcome_mask]

## 3. Clinical table inspection

In [ ]:
clinical.head(3)

In [ ]:
dataframe_information(clinical, name="clinical — dat.csv")

## 4. Medication table inspection

In [ ]:
meds.head(3)

In [ ]:
dataframe_information(meds, name="medications — dat_md.csv")

### 4a. Unique drug names in medication table

Critical to inspect before pivoting — names may have inconsistent spelling,
mixed case, or trailing whitespace that will cause grouping errors in SQL.

In [ ]:
drug_col = meds.columns[1]  # second column is typically drug name
print(f"Drug name column: '{drug_col}'")
print(f"Unique drugs: {meds[drug_col].nunique()}")
print()
print(meds[drug_col].value_counts().to_string())

In [ ]:
# Check for whitespace or case inconsistencies
print("Raw unique values (exact):")
for val in sorted(meds[drug_col].dropna().unique()):
    print(f"  '{val}'")

## 5. Patient ID consistency

Verify the patient identifier links correctly between the two tables.
Any patients in medications but not clinical (or vice versa) need to be
understood before joining.

In [ ]:
id_col_clinical = clinical.columns[0]
id_col_meds = meds.columns[0]

clinical_ids = set(clinical[id_col_clinical].unique())
meds_ids = set(meds[id_col_meds].unique())

print(f"Clinical patient ID column : '{id_col_clinical}'")
print(f"Medication patient ID column: '{id_col_meds}'")
print()
print(f"Unique patients in clinical : {len(clinical_ids)}")
print(f"Unique patients in meds     : {len(meds_ids)}")
print()
print(f"Patients in both tables     : {len(clinical_ids & meds_ids)}")
print(f"In clinical only (no meds)  : {len(clinical_ids - meds_ids)}")
print(f"In meds only (no clinical)  : {len(meds_ids - clinical_ids)}")

For some reason we have one patient in clinical table that is not in the medication table. Lets discover why.

In [ ]:
# Discover the patient id that is missing in meds
missing_ids = clinical.loc[~clinical['inpatient.number'].isin(meds['inpatient.number'])]
missing_ids

## 6. Missingness heatmap

Visualize which columns have missing data and whether missingness
clusters among certain patients.

In [ ]:
missing_pct = clinical.isnull().mean().sort_values(ascending=False)
missing_pct = missing_pct[missing_pct > 0]

if missing_pct.empty:
    print("No missing values in clinical table.")
else:
    fig, ax = plt.subplots(figsize=(10, max(4, len(missing_pct) * 0.3)))
    missing_pct.plot(kind="barh", ax=ax, color="tomato")
    ax.set_xlabel("Fraction missing")
    ax.set_title("Missingness by column — clinical table")
    ax.axvline(0.05,  color="orange", linestyle="--", linewidth=1, label="5%")
    ax.axvline(0.20,  color="red",    linestyle="--", linewidth=1, label="20%")
    ax.legend()
    plt.tight_layout()
    #plt.savefig("../notebooks/figures/01_missingness.png", dpi=100)
    plt.show()
    print(f"\nColumns with >20% missing: "
          f"{missing_pct[missing_pct > 0.20].index.tolist()}")

## 7. Outcome variable

Identify and inspect all potential outcome columns: mortality,
readmission, and follow-up timing.

In [ ]:
outcome_candidates = [c for c in clinical.columns if any(
    kw in c.lower() for kw in
    ["death", "mort", "outcome", "re_adm", "readmit", "follow", "discharge"]
)]

print(f"Outcome candidate columns: {outcome_candidates}")
print()

for col in outcome_candidates:
    print(f"--- {col} ---")
    print(clinical[col].value_counts(dropna=False))
    print()

## 8. Clinical plausibility checks

Verify numeric variables are within physiologically plausible ranges.
Values outside these bounds are likely data entry errors.

In [ ]:
# Claude
plausibility = {

    # ── Vitals ──────────────────────────────────────────────
    "body.temperature":             (33.0,  42.0),
    "pulse":                        (20,    300),
    "respiration":                  (4,     60),
    "systolic.blood.pressure":      (50,    300),
    "diastolic.blood.pressure":     (20,    200),
    "map":                          (30,    200),

    # ── Anthropometrics ─────────────────────────────────────
    "weight":                       (20,    300),
    "height":                       (0.3,   2.5),   # meters
    "BMI":                          (10,    70),

    # ── Cardiac imaging ─────────────────────────────────────
    "LVEF":                         (5,     85),
    "left.ventricular.end.diastolic.diameter.LV": (20, 100),
    "mitral.valve.EMS":             (0.2,   3.0),
    "mitral.valve.AMS":             (0.1,   2.5),
    "EA":                           (0.1,   10.0),
    "tricuspid.valve.return.velocity":  (1.0, 5.0),
    "tricuspid.valve.return.pressure":  (5,   100),

    # ── Neurological ────────────────────────────────────────
    "GCS":                          (3,     15),

    # ── Blood gas ───────────────────────────────────────────
    "pH":                           (6.8,   7.8),
    "oxygen.saturation":            (50,    100),
    "partial.oxygen.pressure":      (20,    700),
    "partial.pressure.of.carbon.dioxide": (10, 150),
    "fio2":                         (21,    100),
    "lactate":                      (0.1,   30.0),
    "glucose.blood.gas":            (1.0,   50.0),
    "body.temperature.blood.gas":   (33.0,  42.0),

    # ── Renal ────────────────────────────────────────────────
    "creatinine.enzymatic.method":  (10,    2000),
    "urea":                         (1.0,   100.0),
    "uric.acid":                    (50,    1500),
    "glomerular.filtration.rate":   (1,     200),
    "cystatin":                     (0.3,   10.0),

    # ── Electrolytes ─────────────────────────────────────────
    "sodium":                       (110,   180),
    "potassium":                    (1.5,   9.0),
    "chloride":                     (70,    130),
    "calcium":                      (1.5,   3.5),
    "sodium.ion":                   (110,   180),
    "potassium.ion":                (1.5,   9.0),
    "chloride.ion":                 (70,    130),
    "free.calcium":                 (0.5,   2.0),
    "serum.magnesium":              (0.3,   3.0),
    "Inorganic.Phosphorus":         (0.3,   4.0),

    # ── Full blood count ─────────────────────────────────────
    "hemoglobin":                   (29,    200),    # g/L
    "white.blood.cell":             (0.5,   100),
    "red.blood.cell":               (1.0,   10.0),
    "platelet":                     (10,    1500),
    "hematocrit":                   (0.05,  0.70),  # proportion
    "neutrophil.count":             (0.1,   50),
    "lymphocyte.count":             (0.1,   20),
    "monocyte.count":               (0.0,   5.0),
    "eosinophil.count":             (0.0,   5.0),
    "basophil.count":               (0.0,   1.0),
    "neutrophil.ratio":             (0,     100),
    "monocyte.ratio":               (0,     30),
    "eosinophil.ratio":             (0,     30),
    "basophil.ratio":               (0,     5),
    "mean.corpuscular.volume":      (50,    150),
    "mean.hemoglobin.volume":       (10,    50),
    "mean.hemoglobin.concentration":(200,   400),
    "mean.platelet.volume":         (5,     20),

    # ── Coagulation ──────────────────────────────────────────
    "international.normalized.ratio":           (0.5,  15.0),
    "activated.partial.thromboplastin.time":    (15,   200),
    "thrombin.time":                            (10,   120),
    "prothrombin.time.ratio":                   (0.5,  5.0),
    "prothrombin.activity":                     (5,    200),
    "fibrinogen":                               (0.5,  15.0),
    "D.dimer":                                  (0.0,  100.0),

    # ── Cardiac biomarkers ───────────────────────────────────
    "brain.natriuretic.peptide":    (0,     50000),
    "high.sensitivity.troponin":    (0,     50000),
    "myoglobin":                    (0,     50000),
    "creatine.kinase":              (10,    50000),
    "creatine.kinase.isoenzyme":    (0,     1000),
    "lactate.dehydrogenase":        (50,    10000),
    "hydroxybutyrate.dehydrogenase":(50,    5000),

    # ── Liver function ───────────────────────────────────────
    "albumin":                      (10,    60),
    "total.protein":                (30,    100),
    "globulin":                     (5,     60),
    "total.bilirubin":              (1,     500),
    "direct.bilirubin":             (0,     300),
    "indirect.bilirubin":           (0,     300),
    "glutamic.pyruvic.transaminase":(5,     5000),
    "glutamic.oxaloacetic.transaminase": (5, 10000), # widened — cardiac events cause extreme elevation
    "alkaline.phosphatase":         (10,    2000),
    "glutamyltranspeptidase":       (5,     2000),
    "cholinesterase":               (500,   20000),
    "total.bile.acid":              (0,     300),

    # ── Lipids ───────────────────────────────────────────────
    "cholesterol":                  (0.5,   20.0),
    "triglyceride":                 (0.1,   50.0),
    "high.density.lipoprotein.cholesterol": (0.1, 5.0),
    "low.density.lipoprotein.cholesterol":  (0.1, 15.0),
    "apolipoprotein.A":             (0.1,   5.0),
    "apolipoprotein.B":             (0.1,   5.0),
    "homocysteine":                 (1,     200),

    # ── Inflammatory ─────────────────────────────────────────
    "high.sensitivity.protein":     (0,     500),
    "erythrocyte.sedimentation.rate": (0,   150),

}

In [ ]:
def plausibility_check(
    df:            pd.DataFrame,
    plausibility:  dict,
) -> pd.DataFrame:
    """
    Check numeric columns against physiologically plausible ranges.
    Returns a summary DataFrame of violations.
    """
    results = []

    for col, (lo, hi) in plausibility.items():
        if col not in df.columns:
            continue

        series      = pd.to_numeric(df[col], errors="coerce")
        n_total     = series.notna().sum()
        violations  = series[(series < lo) | (series > hi)]
        n_violations = len(violations)

        results.append({
            "column":        col,
            "range":         f"[{lo}, {hi}]",
            "n_values":      n_total,
            "n_violations":  n_violations,
            "pct_violations": round(n_violations / n_total * 100, 1) if n_total > 0 else 0,
            "observed_min":  round(series.min(), 2) if n_total > 0 else None,
            "observed_max":  round(series.max(), 2) if n_total > 0 else None,
        })

    summary = pd.DataFrame(results)
    violations_only = summary[summary["n_violations"] > 0].sort_values(
        "pct_violations", ascending=False
    )

    if violations_only.empty:
        print("All columns within plausible ranges.")
    else:
        print(f"{len(violations_only)} column(s) have out-of-range values:\n")
        print(violations_only.to_string(index=False))

    return summary


plausibility_summary = plausibility_check(clinical, plausibility)

In [ ]:
zero_as_missing = [
    "pulse",
    "respiration",
    "systolic.blood.pressure",
    "diastolic.blood.pressure",
    "map",
    "BMI",
    "weight",
]

print("Zero counts in impossible-zero columns:\n")
for col in zero_as_missing:
    n_zeros = (clinical[col] == 0).sum()
    print(f"  {col}: {n_zeros} zeros")

In [ ]:
# Suggested range adjustments presented by Claude
range_updates = {
    "tricuspid.valve.return.pressure": (1.0, 100),   # min was 5, observed 1.70 — plausible in low-pressure states
    "tricuspid.valve.return.velocity": (0.8, 6.0),   # observed 0.90 and 5.76 — borderline, widen slightly
    "glutamic.pyruvic.transaminase": (3, 5000),  # observed min 3.0 — near-zero ALT seen in severe illness
    "prothrombin.activity": (3, 200), # observed 3.80 — extreme coagulopathy
    "thrombin.time": (9, 250), # observed 9.70 and 209.30 — DIC is common in severe HF
    "glomerular.filtration.rate": (1,    300),   # observed 281 — post-dialysis patient possible
    "globulin": (5,    90), # observed 88.30 — extreme but seen in inflammatory states
    "D.dimer": (0,    110), # observed 100.10 — just over boundary, widen ceiling
    "total.protein": (30,   105), # observed 100.90 — just over boundary
    "lymphocyte.count": (0.05, 20), # observed 0.08 — severe lymphopenia in HF
    "red.blood.cell": (0.8,  10.0),  # observed 0.89 — severe anemia
    "prothrombin.time.ratio": (0.5,  16.0),  # observed 14.89 — severe coagulopathy/liver failure
    "international.normalized.ratio": (0.5,  17.0),  # observed 16.59 — same reason
    "cystatin": (0.2,  11.0),  # borderline outliers at 0.23 and 10.37
    "glucose.blood.gas": (0.2,  50.0),  # observed 0.20 — hypoglycemia possible
    "pH": (6.75, 7.8),   # observed 6.77 — severe acidosis, clinically real
    "sodium": (105,  180),   # observed 107.50 — severe hyponatremia, real
    "sodium.ion": (105,  180),   # same
    "calcium": (1.3,  3.5),   # observed 1.39 — severe hypocalcemia, real
    "eosinophil.count": (0.0,  7.0),   # observed 6.58 — eosinophilia, plausible
    "platelet": (5,    1500),  # observed 5 — critical thrombocytopenia, real
    "high.density.lipoprotein.cholesterol": (0.02, 5.0), # observed 0.02 — extremely low but possible
    "oxygen.saturation": (25,   100),  # observed 25 — inspect these rows individually
    "Inorganic.Phosphorus": (0.3,  5.0),  # observed 4.26 — just over, widen ceiling
    "mitral.valve.EMS": (0.01, 3.0), # Prevent borderline cases
    "mitral.valve.AMS": (0.01, 2.5), # Capture severely reduced annular motion
    "EA": (0.01, 10.0), 
}

# Apply updates to plausibility dict
for col, new_range in range_updates.items():
    plausibility[col] = new_range

print("Plausibility ranges updated.")

In [ ]:
# Rerun the plausibility check 
plausibility_summary = plausibility_check(clinical, plausibility)

There seems to be some genuine data errors. Its not possible to have a BME of 0 for example. It also seems the mitral valve ems has 19 violations. 

In [ ]:
genuine_errors = {
    "mitral.valve.EMS":                        409.00,
    "mitral.valve.AMS":                        408.00,
    "EA":                                       21.30,
    "BMI":                                     404.08,
    "left.ventricular.end.diastolic.diameter.LV": 0.30,
    "potassium":                                11.10,
}

print("Genuine data error rows:\n")
for col, suspicious_val in genuine_errors.items():
    lo, hi = plausibility[col]
    rows = clinical[
        (clinical[col] < lo) | (clinical[col] > hi)
    ]
    if not rows.empty:
        print(f"--- {col} ---")
        print(f"  Expected range: [{lo}, {hi}]")
        print(f"  Suspicious values:")
        print(rows[[col]].to_string(index=False))
        print()

* `mitral.valve.EMS / mitral.valve.AMS` at ~409 — almost certainly decimal point entry errors (4.09 intended most likely). Replace with NaN, can't guess the correction
* `EA at 21.30` — likely 2.13 with a decimal error. Replace with NaN
* `BMI at 404.08 or 0` — impossible, replace with NaN
* `left.ventricular.end.diastolic.diameter.LV` at 0.30 — likely stored in cm for most rows but meters here. Replace with NaN
* `potassium` at 11.10 — incompatible with life, replace with NaN

## 9. Outcome stratification

Quick visual check: do the clinical variables separate outcomes as
expected? LVEF and Killip grade should be the strongest separators.
This is purely exploratory — no statistical testing yet.

In [ ]:
# Inspect all candidate columns first
print("Outcome candidate columns found:")
for col in outcome_candidates:
    print(f"\n  {col}:")
    print(f"  {clinical[col].value_counts(dropna=False).to_dict()}")

In [ ]:
outcome_col = outcome_candidates[0]  # use first identified outcome column

key_vars = [c for c in clinical.columns if any(
    kw in c.lower() for kw in ["lvef", "killip", "age", "heartrate", "nyha"]
)][:4]  # limit to first 4 found

if key_vars:
    fig, axes = plt.subplots(1, len(key_vars), figsize=(4 * len(key_vars), 4))
    if len(key_vars) == 1:
        axes = [axes]

    for ax, col in zip(axes, key_vars):
        for outcome_val in clinical[outcome_col].dropna().unique():
            subset = clinical[clinical[outcome_col] == outcome_val][col].dropna()
            ax.hist(subset, bins=20, alpha=0.6, label=str(outcome_val))
        ax.set_title(col)
        ax.set_xlabel(col)
        ax.legend(title=outcome_col)

    plt.suptitle("Key variable distributions by outcome", y=1.02)
    plt.tight_layout()
    # plt.savefig("../notebooks/figures/01_outcome_stratification.png",
    #             dpi=100, bbox_inches="tight")
    plt.show()

# 10. Medication records per patient 

In [ ]:
med_counts = meds.groupby(id_col_meds).size().reset_index(name="n_med_records")

print(med_counts["n_med_records"].describe().round(2))
print()

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(med_counts["n_med_records"], bins=30, color="steelblue", edgecolor="white")
ax.set_title("Medication records per patient")
ax.set_xlabel("Number of records")
ax.set_ylabel("Number of patients")
plt.tight_layout()
#plt.savefig("../notebooks/figures/01_med_records_per_patient.png", dpi=100)
plt.show()

# 12. Correlation with Outcome

In [ ]:
outcome_col = "outcome.during.hospitalization"

# Map outcomes to integers
outcome_numeric = clinical[outcome_col].map({
    "Alive": 0,
    "Dead":  1,
    # "Discharge against orders" becomes NaN — excluded from correlation
})

print(f"Alive   : {(outcome_numeric == 0).sum()}")
print(f"Dead    : {(outcome_numeric == 1).sum()}")
print(f"Excluded: {outcome_numeric.isna().sum()}")

numeric_cols = clinical.select_dtypes(include=[np.number]).columns.tolist()

correlations = (
    clinical[numeric_cols]
    .corrwith(outcome_numeric)
    .dropna()
    .sort_values()
)

print("Most negatively correlated (higher value → more likely Alive):")
print(correlations.head(10).round(3).to_string())

print("\nMost positively correlated (higher value → more likely Dead):")
print(correlations.tail(10).round(3).to_string())

# Get top 20 by absolute value but keep the sign
top20 = (
    correlations
    .abs()
    .sort_values(ascending=False)
    .head(20)
    .index
)

top20_signed = correlations[top20].sort_values()

# Color by direction
colors = ["tomato" if v > 0 else "steelblue" for v in top20_signed]

fig, ax = plt.subplots(figsize=(9, 7))
top20_signed.plot(kind="barh", ax=ax, color=colors)

ax.set_title("Top 20 features — correlation with outcome\n(Dead=1, Alive=0)")
ax.set_xlabel("Pearson correlation")
ax.axvline(0, color="black", linewidth=0.8)

# Add a simple legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="tomato",    label="Positive — higher value → more likely Dead"),
    Patch(facecolor="steelblue", label="Negative — higher value → more likely Alive"),
]
ax.legend(handles=legend_elements, loc="lower right", fontsize=9)

plt.tight_layout()
#plt.savefig("../notebooks/figures/01_outcome_correlations.png", dpi=100)
plt.show()

## 10. Outcome column inventory

The Zigong dataset has multiple outcome columns covering different
time horizons. Document all of them here for use in notebooks
(modeling) and (survival analysis).

In [ ]:
outcome_inventory = [
    "outcome.during.hospitalization",
    "death.within.28.days",
    "re.admission.within.28.days",
    "death.within.3.months",
    "re.admission.within.3.months",
    "death.within.6.months",
    "re.admission.within.6.months",
    "time.of.death..days.from.admission.",
    "re.admission.time..days.from.admission.",
    "return.to.emergency.department.within.6.months",
    "time.to.emergency.department.within.6.months",
]

print("Outcome column summary:\n")
for col in outcome_inventory:
    if col not in clinical.columns:
        print(f"  {col}: NOT FOUND")
        continue
    n_missing = clinical[col].isna().sum()
    val_counts = clinical[col].value_counts(dropna=False).to_dict()
    print(f"  {col}")
    print(f"    missing : {n_missing}")
    print(f"    values  : {val_counts}")
    print()

# 14. Survival Time Columns

In [ ]:
time_cols = [
    "time.of.death..days.from.admission.",
    "re.admission.time..days.from.admission.",
    "time.to.emergency.department.within.6.months",
    "dischargeDay",
]

print("Time variable summary:\n")
for col in time_cols:
    if col not in clinical.columns:
        print(f"  {col}: NOT FOUND")
        continue
    series = pd.to_numeric(clinical[col], errors="coerce")
    print(f"  {col}")
    print(f"    min    : {series.min():.1f} days")
    print(f"    max    : {series.max():.1f} days")
    print(f"    median : {series.median():.1f} days")
    print(f"    missing: {series.isna().sum()}")
    print()

## 11. Inspection findings summary

Fill in after running all cells. This is the to-do list for
`02_data_cleaning.ipynb`.

### Schema
- [ ] Patient ID column: ___
- [ ] Primary outcome column confirmed: `outcome.during.hospitalization`
- [ ] Class distribution (0 vs 1): ___

### Unit encoding
- [ ] `height` stored in: meters / cm (circle one)
- [ ] `hematocrit` stored as: proportion / percentage (circle one)
- [ ] `hemoglobin` stored in: g/L / g/dL (circle one)

### Missing data
- [ ] Columns with >20% missing: ___
- [ ] Columns with 5–20% missing: ___
- [ ] Columns with <5% missing: ___

### Zeros as missing
- [ ] Confirmed zero-as-missing columns: `pulse`, `respiration`,
      `systolic.blood.pressure`, `diastolic.blood.pressure`,
      `map`, `BMI`, `weight`

### Genuine data errors (replace with NaN in notebook 02)
- [ ] `mitral.valve.EMS` > 3.0
- [ ] `mitral.valve.AMS` > 2.5
- [ ] `EA` > 10.0
- [ ] `BMI` > 70
- [ ] `left.ventricular.end.diastolic.diameter.LV` < 1.0
- [ ] `potassium` > 9.0

### Drug name issues in medication table
- [ ] Mixed case: yes / no
- [ ] Trailing whitespace: yes / no
- [ ] Unexpected drug names: ___

### Survival analysis columns confirmed
- [ ] Time-to-event column: ___
- [ ] Event indicator column: ___
- [ ] Follow-up horizons available: 28d / 3m / 6m

### Notes for notebook 02
- ___